# Cross-lead (lead-conditioned) downscaling — analysis

**Paper experiment §3.3.3 P1 — lead-conditioned variant.** Unlike the zero-shot
study (train on ERA5 only, evaluate zero-shot on Aurora), here a **single model
per config is trained jointly on all four leads** via `--lead-datasets`, with a
normalised `lead/72` context channel so the heads can learn a lead-dependent
predictive spread σ(lead). The downscaling target (station obs at the valid
time) is identical across leads — the leads are just alternative coarse inputs,
so one epoch sees every episode at every lead.

Each trained model is evaluated **per lead** (the eval sets `--lead-hours` to
match the eval dataset, so the lead channel is correct):

| eval dir | context grid | lead | role |
|---|---|---|---|
| `eval_lead0h`  | real ERA5 (precip dropped) | 0  | shortest lead / in-distribution anchor |
| `eval_lead6h`  | Aurora forecast | +6h  | trained lead |
| `eval_lead24h` | Aurora forecast | +24h | trained lead |
| `eval_lead72h` | Aurora forecast | +72h | trained lead |

**Matrix:** 4 configs × {europe, east_asia} × seeds {42, 123, 456}, each with the
4 per-lead evals. Configs cover two targets and **two** variants (all NLL):

- **t2m** — Gaussian head. **ConvCNP (no TESSERA)** (bilinear, + static + elev)
  vs **ConvCNP with TESSERA** (raw 16-d VAE latent, no static, + elev).
- **wind** — Truncated-Normal head. Same two variants.

A **naive ERA5 bilinear-interpolation floor** (§1b) is overlaid on every skill-vs-lead plot, headline table and uplift plot as a no-model reference.

**Scientific questions this notebook answers**

1. How does skill (RMSE / MAE / CRPS) change with lead for a model that has *seen*
   every lead in training — i.e. the genuine information-loss curve of the
   forecast context, not a distribution shift?
2. Does the lead channel let the head **widen σ with lead** (a learned σ(lead))?
   If so, calibration should hold across leads — unlike the frozen-spread
   zero-shot case.
3. Does the TESSERA latent help (`concat` vs `baseline`), and does its
   advantage hold as the coarse context degrades with lead?

**Inputs.** Reads only `test_summary.json`, `test_predictions.npz`, and
`test_station_errors.npz` written by `evaluate.py` — no torch / dataset code,
so it runs anywhere with numpy / pandas / matplotlib / scipy.

In [ ]:
# === Configuration =========================================================
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

%matplotlib inline
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 140,
        "figure.autolayout": True,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.size": 10,
    }
)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

# Cross-lead training-runs root (per-region folders live under it), resolved
# under $TESSERA_DATA_ROOT.
from tessera_downscaling.paths import training_runs_dir

RESULTS_ROOT = training_runs_dir("snapshot_14y_cross_lead")

REGIONS = ["europe", "east_asia"]
SEEDS = [42, 123, 456]

# eval subdir -> (pretty label, lead hours). lead 0 = real ERA5 analysis (anchor).
EVAL_SOURCES = {
    "eval_lead0h": ("ERA5 (lead 0)", 0),
    "eval_lead6h": ("Lead +6h", 6),
    "eval_lead24h": ("Lead +24h", 24),
    "eval_lead72h": ("Lead +72h", 72),
}
EVAL_ORDER = [v[0] for v in EVAL_SOURCES.values()]  # categorical x-axis order
LEAD_HOURS = [v[1] for v in EVAL_SOURCES.values()]
FORECAST_LABELS = EVAL_ORDER[1:]  # the +6/+24/+72h leads

# Variant axis (replaces the zero-shot model/loss axes; loss is always NLL here).
VARIANT_ORDER = ["baseline", "concat"]
VARIANT_COLORS = {"baseline": "#888888", "concat": "#1f77b4"}

# Where figures/CSVs get written.
OUT_DIR = Path("cross_lead_analysis_outputs")
OUT_DIR.mkdir(exist_ok=True)
print("RESULTS_ROOT:", RESULTS_ROOT, "\nexists:", RESULTS_ROOT.exists())

# --- Paper-facing variant display names -------------------------------------
# concat is the sole TESSERA fusion in this experiment, so refer to the two
# ConvCNP variants by whether they use TESSERA rather than by fusion mechanism.
VARIANT_DISPLAY = {"baseline": "ConvCNP (no TESSERA)", "concat": "ConvCNP with TESSERA"}


def vlabel(v):
    return VARIANT_DISPLAY.get(v, v)


# --- ERA5 bilinear-interpolation baseline (naive per-lead floor) -------------
# Not a ConvCNP: at each lead we bilinearly interpolate the SAME coarse context
# the model was given (real ERA5 at lead 0; the Aurora forecast at +6/+24/+72h)
# to the station, with a constant residual-sigma predictive spread. Computed by
# tessera_downscaling.baselines (tessera-baselines) on BOTH the full station set
# (comparable to the no-TESSERA ConvCNP) and the matched TESSERA-inter-VAE set
# (comparable to the TESSERA ConvCNP). station_set -> (config-dir stem, comparable variant).
INTERP_SETS = {
    "full": ("era5_interp_baseline_full", "baseline"),
    "matched": ("era5_interp_baseline_matched", "concat"),
}
INTERP_COLOR = "#111111"
INTERP_DISPLAY = "ERA5 bilinear interp"

In [ ]:
# === TESSERA generation selector ===========================================
# Pick which latents generation the TESSERA concat arm loads — every cell
# below follows this choice. The concat config was retrained per generation
# with identical args (only the latents file differs); the no-TESSERA
# baseline and ERA5-interp rows are latents-independent and always come from
# RESULTS_ROOT.

# --- current main: TESSERA v2 "1B-M", 2017 embeddings (crop64_lat16_auxon) ---
TESSERA_RESULTS_ROOT = training_runs_dir("snapshot_14y_cross_lead_tessera_1B-M_2017")

# --- previous main: TESSERA v1 16-d latents (uncomment to switch back; the
#     concat arm then loads from the same folder as the baselines) -----------
# TESSERA_RESULTS_ROOT = RESULTS_ROOT

print(
    "TESSERA_RESULTS_ROOT:",
    TESSERA_RESULTS_ROOT,
    "\nexists:",
    TESSERA_RESULTS_ROOT.exists(),
)

## 1. Load every `test_summary.json` into one tidy table

One row per `(region, config, seed, eval_source, variable)`. The config name is
parsed into `target` / `variant` (and a derived `is_tessera`) so we can group on
them. `loss` is omitted — every cross-lead config trains on NLL.

In [ ]:
# === Parse + load ==========================================================
def parse_config(name: str) -> dict:
    """cross-lead config dir name -> structured factors."""
    target = "t2m" if name.startswith("t2m") else "wind"
    if "era5_interp" in name:
        variant = "era5_interp"  # naive floor; excluded from model comparisons
    elif "concat" in name:
        variant = "concat"
    elif "film" in name:
        variant = "film"
    else:
        variant = "baseline"
    head = "gaussian" if target == "t2m" else "truncated_normal"
    return {
        "target": target,
        "variant": variant,
        "head": head,
        "is_tessera": variant in ("concat", "film"),
    }


# Metrics shared by the Gaussian (t2m) and Truncated-Normal (wind) heads, named
# identically by evaluate.py, plus the universal proper-scoring metrics.
SCALAR_METRICS = [
    "nll",
    "crps",
    "mae",
    "rmse",
    "bias",
    "correlation",
    "mean_pred_std",
    "within_1sigma",
    "within_2sigma",
    "pit_chi2_stat",
    "pit_chi2_pvalue",
    "p50",
    "p90",
    "p95",
    "p99",
    "n_predictions",
    "n_test_stations",
]
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# evaluate.py names the point-estimate metrics by the functional the head uses:
# the Gaussian (t2m) head reports them plainly (point estimate = mean), while the
# Truncated-Normal (wind) head reports RMSE/bias/correlation @ the mean and MAE @
# the median (the proper point estimate for a [0, inf) variable). Map both onto
# the canonical column names so the rest of the notebook stays head-agnostic.
POINT_METRIC_KEYS = {
    "gaussian": {},  # plain {var}_mae / _rmse / _bias / _correlation
    "truncated_normal": {
        "mae": "mae_at_median",
        "rmse": "rmse_at_mean",
        "bias": "bias_at_mean",
        "correlation": "correlation_at_mean",
    },
}


def g(d, key, default=np.nan):
    v = d.get(key, default)
    return v if v is not None else default


rows = []
_scan = [(r, RESULTS_ROOT / r, "base") for r in REGIONS]
if TESSERA_RESULTS_ROOT != RESULTS_ROOT:
    _scan += [(r, TESSERA_RESULTS_ROOT / r, "tessera") for r in REGIONS]
for region, region_dir, _role in _scan:
    if not region_dir.is_dir():
        continue
    for seed_dir in sorted(region_dir.glob("*_seed*")):
        m = re.match(r"(?P<config>.+)_seed(?P<seed>\d+)$", seed_dir.name)
        if not m:
            continue
        config = m.group("config")
        seed = int(m.group("seed"))
        factors = parse_config(config)
        # With a generation-specific TESSERA root, the TESSERA arm comes
        # exclusively from there and everything else (baseline / era5_interp)
        # exclusively from RESULTS_ROOT — old-root TESSERA runs are skipped.
        if (
            TESSERA_RESULTS_ROOT != RESULTS_ROOT
            and (_role == "tessera") != factors["is_tessera"]
        ):
            continue
        for eval_key, (eval_label, lead) in EVAL_SOURCES.items():
            summ = seed_dir / eval_key / "test_summary.json"
            present = summ.exists()
            base = dict(
                region=region,
                config=config,
                seed=seed,
                eval_source=eval_key,
                eval_label=eval_label,
                lead_hours=lead,
                present=present,
                **factors,
            )
            if not present:
                rows.append({**base, "variable": factors["target"]})
                continue
            d = json.loads(summ.read_text())
            tvars = d.get("target_variables", [factors["target"]])
            for var in tvars:
                row = {
                    **base,
                    "variable": var,
                    "checkpoint_epoch": g(d, "checkpoint_epoch"),
                    "best_val_loss": g(d, "best_val_loss"),
                }
                keymap = POINT_METRIC_KEYS[factors["head"]]
                for met in SCALAR_METRICS:
                    row[met] = g(d, f"{var}_{keymap.get(met, met)}")
                seas = d.get(f"{var}_seasonal_mae", {}) or {}
                for s in SEASONS:
                    row[f"seasonal_mae_{s}"] = g(seas, s)
                rows.append(row)

df = pd.DataFrame(rows)
# Drop variants outside VARIANT_ORDER (e.g. film) so they never enter a
# comparison; their config dirs are still parsed, just excluded here.
df = df[df["variant"].isin(VARIANT_ORDER)].copy()
# Stable categorical ordering for plots/tables.
df["eval_label"] = pd.Categorical(df["eval_label"], categories=EVAL_ORDER, ordered=True)
df["variant"] = pd.Categorical(df["variant"], categories=VARIANT_ORDER, ordered=True)
print(
    f"Loaded {df['present'].sum()} eval cells "
    f"({df[df.present].drop_duplicates(['region', 'config', 'seed', 'eval_source']).shape[0]} unique)."
)
df.head()

### 1a. Completeness — which of the expected cells are present?

Expect `4 configs × {2 regions} × {3 seeds} × {4 leads} = 96` cells. Missing
cells (unfinished or failed jobs) are listed so nothing silently drops out — and
so a half-finished sweep is read honestly while training is still underway.

In [ ]:
# === Completeness ==========================================================
all_configs = sorted(df["config"].unique())
expected = len(all_configs) * len(REGIONS) * len(SEEDS) * len(EVAL_SOURCES)
got = int(df["present"].sum())
print(f"Present {got} / {expected} expected eval cells.\n")

comp = (
    df.assign(ok=df["present"].astype(int))
    .pivot_table(
        index=["region", "config", "seed"],
        columns="eval_label",
        values="ok",
        aggfunc="max",
        observed=False,
    )
    .reindex(columns=EVAL_ORDER)
)
missing = comp[(comp.fillna(0) == 0).any(axis=1)]
if len(missing):
    print("Incomplete (run dir, missing leads shown as 0/NaN):")
    display(missing)
else:
    print("All expected cells present. \u2713")
# keep only present rows for the analyses below
D = df[df["present"]].copy()

### 1b. ERA5 bilinear-interpolation baseline (naive floor)

A no-model reference: at each lead, bilinearly interpolate the coarse context the ConvCNP saw (real ERA5 at lead 0; the Aurora forecast at +6/+24/+72h) straight to the station. Computed on **both** the full station set (comparable to the no-TESSERA ConvCNP) and the matched TESSERA∩VAE set (comparable to the TESSERA ConvCNP). Every skill-vs-lead plot, headline table and uplift plot below shows this floor.

In [ ]:
# === Load the ERA5-interp baselines (naive per-lead floor) =================
# One row per (baseline_kind, region, target, station_set, eval_label). Reads the
# deterministic era5_interp runs computed alongside the ConvCNPs. Their summary has
# no CRPS, so we add the Gaussian closed-form CRPS of the constant-sigma predictive
# (mean = interpolated field, sigma = residual std), consistent with how the
# baseline already defines its own NLL / coverage.
#
# TWO reference kinds are loaded:
#   "interp"        plain bilinear interpolation of the coarse context (both vars).
#   "interp_lapse"  the same interpolation with a fitted lapse-rate correction
#                   T_interp - Gamma*dElev applied. t2m ONLY -- the correction is
#                   t2m-only by design in tessera-baselines, so no wind
#                   rows exist and none are expected.
# INTERP keeps ONLY the plain rows, so every cell that already uses it (headline
# table, skill-vs-lead floor, interp_series) behaves exactly as before. The
# corrected rows land in INTERP_LAPSE for the cells that want the stronger
# temperature reference.
from scipy.stats import norm as _NORM

INTERP_KIND_STEMS = {
    "interp": "era5_interp_baseline",
    "interp_lapse": "era5_interp_lapse_baseline",
}


def _gauss_crps_mean(pred, y, sigma):
    """Mean CRPS of N(pred, sigma^2) vs y (closed form)."""
    sigma = np.asarray(sigma, float)
    z = (np.asarray(y, float) - np.asarray(pred, float)) / sigma
    per = sigma * (z * (2 * _NORM.cdf(z) - 1) + 2 * _NORM.pdf(z) - 1.0 / np.sqrt(np.pi))
    return float(np.mean(per))


interp_rows = []
for kind, stem_base in INTERP_KIND_STEMS.items():
    for region in REGIONS:
        for target in ["t2m", "wind"]:
            for sset, (_stem, cmp_variant) in INTERP_SETS.items():
                cfg = f"{target}_snap_{stem_base}_{sset}_seed42"
                for eval_key, (eval_label, lead) in EVAL_SOURCES.items():
                    d_dir = RESULTS_ROOT / region / cfg / eval_key
                    summ = d_dir / "test_summary.json"
                    if not summ.exists():
                        continue
                    d = json.loads(summ.read_text())
                    row = dict(
                        baseline_kind=kind,
                        region=region,
                        target=target,
                        variable=target,
                        station_set=sset,
                        cmp_variant=cmp_variant,
                        eval_source=eval_key,
                        eval_label=eval_label,
                        lead_hours=lead,
                        rmse=g(d, f"{target}_rmse"),
                        mae=g(d, f"{target}_mae"),
                        bias=g(d, f"{target}_bias"),
                        mean_pred_std=g(d, f"{target}_mean_pred_std"),
                        n_predictions=g(d, f"{target}_n_predictions"),
                        n_test_stations=g(d, f"{target}_n_test_stations"),
                        crps=np.nan,
                        # Gamma in K/km, so the applied correction is auditable
                        # from the frame rather than only from the run logs.
                        lapse_k_per_km=(d.get("lapse_rate") or {}).get(
                            "gamma_k_per_km", np.nan
                        ),
                    )
                    npz = d_dir / "test_predictions.npz"
                    if npz.exists():
                        with np.load(npz, allow_pickle=True) as z:
                            if f"{target}_predictions" in z.files:
                                p = z[f"{target}_predictions"].astype(np.float64)
                                yv = z[f"{target}_targets"].astype(np.float64)
                                sg = z[f"{target}_predicted_stds"].astype(np.float64)
                                ok = (
                                    np.isfinite(p)
                                    & np.isfinite(yv)
                                    & np.isfinite(sg)
                                    & (sg > 0)
                                )
                                if ok.any():
                                    row["crps"] = _gauss_crps_mean(
                                        p[ok], yv[ok], sg[ok]
                                    )
                    interp_rows.append(row)

INTERP_ALL = pd.DataFrame(interp_rows)
if not INTERP_ALL.empty:
    INTERP_ALL["eval_label"] = pd.Categorical(
        INTERP_ALL["eval_label"], categories=EVAL_ORDER, ordered=True
    )
INTERP = (
    INTERP_ALL[INTERP_ALL.baseline_kind == "interp"].copy()
    if not INTERP_ALL.empty
    else INTERP_ALL
)
INTERP_LAPSE = (
    INTERP_ALL[INTERP_ALL.baseline_kind == "interp_lapse"].copy()
    if not INTERP_ALL.empty
    else INTERP_ALL
)

if not INTERP.empty:
    print(
        f"Loaded {len(INTERP)} plain ERA5-interp cells across "
        f"{INTERP.station_set.nunique()} station set(s): {sorted(INTERP.station_set.unique())}."
    )
    display(
        INTERP.groupby(["target", "station_set"], observed=True)["n_test_stations"].agg(
            ["min", "max"]
        )
    )
else:
    print("WARNING: no ERA5-interp baseline runs found under RESULTS_ROOT.")

if not INTERP_LAPSE.empty:
    _lap = INTERP_LAPSE.groupby(["region", "target", "station_set"], observed=True).agg(
        n_cells=("mae", "size"),
        gamma_min=("lapse_k_per_km", "min"),
        gamma_max=("lapse_k_per_km", "max"),
    )
    print(
        f"\nLoaded {len(INTERP_LAPSE)} lapse-corrected ERA5-interp cells "
        "(t2m only by design). Fitted Gamma, K/km:"
    )
    display(_lap.round(3))
else:
    print(
        "\nNOTE: no lapse-corrected ERA5-interp runs found — cells that ask for "
        "the stronger t2m reference will fall back to plain interpolation."
    )


def interp_series(region, target, metric, station_set="matched"):
    """ERA5-interp metric over EVAL_ORDER for one region/target/station set.

    Plain interpolation only, so the floors drawn in the skill-vs-lead panels are
    unchanged by the addition of the lapse-corrected rows.
    """
    if INTERP.empty:
        return pd.Series(index=EVAL_ORDER, dtype=float)
    s = INTERP[
        (INTERP.region == region)
        & (INTERP.target == target)
        & (INTERP.station_set == station_set)
    ]
    if s.empty:
        return pd.Series(index=EVAL_ORDER, dtype=float)
    return s.set_index("eval_label")[metric].reindex(EVAL_ORDER)

## 2. Headline metrics (seed mean ± std)

Per `(region, target, variant)`, the key metrics for each lead side by side.
Deterministic accuracy: **RMSE / MAE / bias / correlation**. Probabilistic:
**CRPS / NLL** (both proper; lower is better).

In [ ]:
# === Aggregation helpers ===================================================
def agg(frame, metric, by=("region", "target", "variant", "eval_label")):
    by = list(by)
    out = (
        frame.groupby(by, observed=True)[metric]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )
    return out


def fmt_ms(m, s):
    if pd.isna(m):
        return ""
    return f"{m:.3f}±{s:.3f}" if pd.notna(s) else f"{m:.3f}"


def mae_label(target):
    """t2m point estimate is the predictive mean; wind (truncated normal) the median."""
    return "MAE@median" if target == "wind" else "MAE@mean"


def _interp_table_rows(target, metric):
    """(region, 'ERA5 bilinear interp (set)') -> {lead: formatted value} for the floor."""
    rows = {}
    if INTERP.empty or metric not in INTERP.columns:  # e.g. NLL: models only
        return rows
    sub = INTERP[INTERP.target == target]
    for (region, sset), grp in sub.groupby(["region", "station_set"], observed=True):
        ser = grp.set_index("eval_label")[metric].reindex(EVAL_ORDER)
        rows[(region, f"{INTERP_DISPLAY} ({sset})")] = {
            lab: fmt_ms(ser.get(lab, np.nan), np.nan) for lab in EVAL_ORDER
        }
    return rows


def pivot_mean_std(frame, metric, target):
    sub = frame[frame.target == target]
    a = agg(sub, metric)
    a["cell"] = [fmt_ms(m, s) for m, s in zip(a["mean"], a["std"], strict=False)]
    a["variant"] = a["variant"].map(vlabel)  # paper-facing model names
    p = a.pivot_table(
        index=["region", "variant"],
        columns="eval_label",
        values="cell",
        aggfunc="first",
        observed=False,
    ).reindex(columns=EVAL_ORDER)
    extra = _interp_table_rows(target, metric)  # append the naive-interp floor rows
    if extra:
        ei = pd.DataFrame.from_dict(extra, orient="index").reindex(columns=EVAL_ORDER)
        ei.index = pd.MultiIndex.from_tuples(ei.index, names=["region", "variant"])
        p = pd.concat([p, ei]).sort_index(kind="stable")
    return p


for target in ["t2m", "wind"]:
    if (D.target == target).sum() == 0:
        continue
    print(
        f"\n{'=' * 70}\n{target.upper()}  —  RMSE (seed mean±std; ERA5-interp is deterministic)\n{'=' * 70}"
    )
    display(pivot_mean_std(D, "rmse", target))
    print(f"{target.upper()}  —  {mae_label(target)}")
    display(pivot_mean_std(D, "mae", target))
    print(f"{target.upper()}  —  CRPS")
    display(pivot_mean_std(D, "crps", target))
    print(f"{target.upper()}  —  NLL  (models only)")
    display(pivot_mean_std(D, "nll", target))

## 3. Skill vs lead (the trained, lead-conditioned curve)

The signature figure. x-axis = lead (ERA5 lead-0 anchor, then +6 / +24 / +72h);
lines = the three variants; markers = seed mean, band = ±1 std across seeds.

Because the model was **trained on every lead**, a rising curve here is the
genuine skill lost to the forecast-context information at longer lead — *not* a
zero-shot distribution shift. The vertical gap between variants is the TESSERA
contribution (and how it changes with lead).

In [ ]:
# === Skill-vs-lead plots ===================================================
def plot_metric_vs_lead(frame, target, metric, ylabel=None):
    sub = frame[frame.target == target]
    if sub.empty:
        print(f"(no {target} data)")
        return
    ncol = len(REGIONS)
    # sharey=True: regions share target+metric units, so a common y-scale makes
    # the decay gradient directly comparable across regions.
    fig, axes = plt.subplots(
        1, ncol, figsize=(5.4 * ncol, 3.8), squeeze=False, sharey=True
    )
    x = np.arange(len(EVAL_ORDER))
    for ci, region in enumerate(REGIONS):
        ax = axes[0][ci]
        cell = sub[sub.region == region]
        for variant in VARIANT_ORDER:
            a = (
                agg(cell[cell.variant == variant], metric)
                .set_index("eval_label")
                .reindex(EVAL_ORDER)
            )
            if a["mean"].notna().sum() == 0:
                continue
            ax.plot(
                x,
                a["mean"].values,
                "-o",
                color=VARIANT_COLORS[variant],
                label=vlabel(variant),
                lw=1.8,
                ms=5,
            )
            lo = a["mean"].values - a["std"].fillna(0).values
            hi = a["mean"].values + a["std"].fillna(0).values
            ax.fill_between(x, lo, hi, color=VARIANT_COLORS[variant], alpha=0.15)
        # Naive ERA5 bilinear-interp floor (matched station set = comparable to the
        # headline TESSERA model; the full-set floor differs by <~1.5%).
        isr = interp_series(region, target, metric, "matched")
        if isr.notna().sum() > 0:
            ax.plot(
                x,
                isr.values,
                ":D",
                color=INTERP_COLOR,
                lw=1.6,
                ms=4,
                label=INTERP_DISPLAY,
            )
        ax.axvline(
            0.5, color="k", ls=":", lw=0.8, alpha=0.5
        )  # lead-0 | forecast divider
        ax.set_xticks(x)
        ax.set_xticklabels(EVAL_ORDER, rotation=20, ha="right")
        ax.set_title(f"{region}  ({target})")
        if ci == 0:
            ax.set_ylabel(ylabel or metric.upper())
        ax.legend(fontsize=8)
    fig.suptitle(
        f"{target.upper()} — {ylabel or metric.upper()} vs lead "
        f"(dotted = naive ERA5 interp floor)",
        y=1.02,
        fontsize=12,
    )
    fig.savefig(OUT_DIR / f"skill_vs_lead_{target}_{metric}.png", bbox_inches="tight")
    plt.show()


for target in ["t2m", "wind"]:
    plot_metric_vs_lead(D, target, "rmse", "RMSE")
    plot_metric_vs_lead(D, target, "mae", mae_label(target))
    plot_metric_vs_lead(D, target, "crps", "CRPS")

In [ ]:
# === Normalised skill decay vs lead ========================================
# Each curve is normalised to its OWN lead-0 anchor (% increase vs lead-0), so
# the y-axis is unitless and t2m (K) and wind (m/s) sit on one shared scale --
# the *slope is the message*: this answers "does t2m decay faster than wind?".
#
# CAVEAT — read this ACROSS VARIABLES (t2m vs wind), not ACROSS VARIANTS
# (baseline vs concat). Because every curve is divided by its *own* lead-0, the
# lower-error variant (concat) divides by a smaller number, so it can show a
# *larger* % even when its degradation in physical units is equal or smaller.
# e.g. t2m/east_asia RMSE: baseline 1.841->2.156 (+0.315 K, +17.1%), concat
# 1.539->1.830 (+0.291 K, +18.9%) -- concat degrades LESS in K yet MORE in %,
# and is still lower error at every lead. So "baseline decays less in %" here is
# NOT in tension with concat's positive uplift; it's the small-denominator
# effect. For the cross-variant question (does TESSERA's edge hold with lead?)
# read the section 5 uplift-vs-lead plots, which compare variants at one scale.
from matplotlib.lines import Line2D

TARGET_COLORS = {
    "t2m": "#d62728",
    "wind": "#1f77b4",
}  # colour  = variable (temp warm / wind cool)
REL_DASH = {"baseline": "--", "concat": "-"}  # line   = variant
REL_MARKER = {
    "baseline": "s",
    "concat": "o",
}  # marker = variant (keeps near-overlapping curves distinct)


def relative_degradation_per_seed(frame, metric):
    """Per (region,target,variant,seed): % increase vs that seed's OWN lead-0 value."""
    keys = ["region", "target", "variant", "seed"]
    piv = frame.pivot_table(
        index=keys, columns="eval_label", values=metric, observed=True
    ).reindex(columns=EVAL_ORDER)
    base = piv[EVAL_ORDER[0]]
    return (
        piv.sub(base, axis=0).div(base, axis=0) * 100.0
    )  # 0% at lead-0 by construction


def plot_relative_vs_lead(frame, metric, ylabel=None):
    rel = relative_degradation_per_seed(frame, metric)
    g = rel.groupby(level=["region", "target", "variant"], observed=True)
    mean, std = g.mean(), g.std()
    targets = [t for t in ["t2m", "wind"] if (frame.target == t).any()]
    fig, axes = plt.subplots(
        1, len(REGIONS), figsize=(5.4 * len(REGIONS), 3.8), squeeze=False, sharey=True
    )
    x = np.arange(len(EVAL_ORDER))
    for ci, region in enumerate(REGIONS):
        ax = axes[0][ci]
        for target in targets:
            for variant in VARIANT_ORDER:
                key = (region, target, variant)
                if key not in mean.index:
                    continue
                m = mean.loc[key].reindex(EVAL_ORDER).values
                s = std.loc[key].reindex(EVAL_ORDER).fillna(0).values
                ax.plot(
                    x,
                    m,
                    ls=REL_DASH[variant],
                    marker=REL_MARKER[variant],
                    color=TARGET_COLORS[target],
                    lw=1.8,
                    ms=5,
                )
                ax.fill_between(
                    x, m - s, m + s, color=TARGET_COLORS[target], alpha=0.12
                )
        ax.axhline(0, color="k", lw=0.8)  # lead-0 anchor (0%)
        ax.axvline(
            0.5, color="k", ls=":", lw=0.8, alpha=0.5
        )  # lead-0 | forecast divider
        ax.set_xticks(x)
        ax.set_xticklabels(EVAL_ORDER, rotation=20, ha="right")
        ax.set_title(f"{region}")
        if ci == 0:
            ax.set_ylabel(ylabel or f"Δ {metric.upper()} vs own lead-0  (%)")
    # Single combined legend (colour = variable, dash+marker = variant), placed in
    # the upper-left of the first panel where the decay curves leave room. One box
    # for a paper -- readers shouldn't have to join two separate keys.
    _hdr = lambda t: Line2D([], [], ls="none", marker="", label=t)
    leg_handles = [_hdr(r"$\bf{variable}$")]
    leg_handles += [
        Line2D([0], [0], color=TARGET_COLORS[t], lw=2.6, label=t) for t in targets
    ]
    leg_handles += [_hdr(""), _hdr(r"$\bf{variant}$")]
    leg_handles += [
        Line2D(
            [0],
            [0],
            color="0.35",
            lw=1.8,
            ls=REL_DASH[v],
            marker=REL_MARKER[v],
            ms=6,
            label=vlabel(v),
        )
        for v in VARIANT_ORDER
    ]
    axes[0][0].legend(
        handles=leg_handles,
        loc="upper left",
        fontsize=8,
        framealpha=0.9,
        handlelength=2.6,
        borderaxespad=0.5,
    )
    # fig.suptitle(f"Relative skill decay vs lead — {metric.upper()} "
    #              f"(each curve normalised to its OWN lead-0; slope = decay rate)",
    #              y=1.02, fontsize=12)
    fig.savefig(OUT_DIR / f"relative_skill_vs_lead_{metric}.png", bbox_inches="tight")
    plt.show()


for metric in ["rmse", "mae", "crps"]:
    plot_relative_vs_lead(D, metric)

## 4. Skill loss vs lead, quantified

For each variant, the **% increase** in error relative to the lead-0 (ERA5)
anchor at each forecast lead. This is the headline "cost of a longer-lead
forecast context" number, for a model that was trained to handle every lead.

In [ ]:
# === % change vs lead-0 anchor =============================================
def degradation_table(frame, metric):
    recs = []
    keys = ["region", "target", "variant"]
    means = (
        agg(frame, metric)
        .pivot_table(index=keys, columns="eval_label", values="mean", observed=False)
        .reindex(columns=EVAL_ORDER)
    )
    anchor = EVAL_ORDER[0]
    for idx, r in means.iterrows():
        base = r.get(anchor, np.nan)
        rec = dict(zip(keys, idx, strict=False))
        rec[anchor] = base
        for lab in FORECAST_LABELS:
            v = r.get(lab, np.nan)
            rec[f"{lab} (\u0394%)"] = (
                (v - base) / base * 100 if (pd.notna(v) and base) else np.nan
            )
        recs.append(rec)
    return pd.DataFrame(recs)


for metric in ["rmse", "mae", "crps"]:
    print(
        f"\n{'=' * 70}\n{metric.upper()} \u2014 % increase vs lead-0 context\n{'=' * 70}"
    )
    t = degradation_table(D, metric)
    fmt = {c: "{:+.1f}%".format for c in t.columns if "\u0394%" in c}
    fmt[EVAL_ORDER[0]] = "{:.3f}".format
    display(t.style.format(fmt).hide(axis="index"))

## 5. TESSERA uplift vs its two references

The reference model here is **always ConvCNP with TESSERA**. For each metric we
plot its uplift (% better; positive = TESSERA wins) over **two** references on the
same axes, split per variable (rows) and per region (columns):

- **vs ConvCNP (no TESSERA)** — the model-vs-model gain from adding the satellite
  embedding (paired per seed).
- **vs the interpolated coarse field** — the gain over the naive no-model floor
  (TESSERA and the interpolation both on the matched TESSERA∩VAE station set).

The second reference is bilinear interpolation of the **same $0.25^\circ$ context
field the model was given at that lead** — real ERA5 at lead 0, the Aurora forecast
at +6/+24/+72h. It is therefore not an "ERA5 floor" beyond lead 0, and is not
labelled as one.

Bilinear interpolation is common to both variables; what differs is the correction
applied on top. For **t2m** we use the *lapse-rate-corrected* interpolation
($T_{\text{interp}} - \Gamma\,\Delta\text{elev}$, $\Gamma$ fitted on the train
split). Uncorrected interpolation carries a systematic elevation bias that a single
fitted constant removes, so scoring against it would credit the model for a
correction that trivial post-processing already makes. **Wind** has no lapse
analogue — $10$ m wind speed does not vary with height at a fixed environmental
rate — so it is scored against the uncorrected field. The legend states which
correction was applied, built from the loaded rows rather than hard-coded.

We deliberately never plot ConvCNP-without-TESSERA vs interp — every curve is
anchored to the TESSERA model. The question is whether both gaps **hold up** as the
forecast context degrades with lead. Legend is below the panels.

In [ ]:
# === TESSERA uplift vs BOTH references (anchored to ConvCNP with TESSERA) ====
# Reference model is always ConvCNP with TESSERA (concat). We show its uplift
# (% better) over two references on one axes, per variable x region:
#   * vs ConvCNP (no TESSERA) -- model-vs-model, paired per seed.
#   * vs the interpolated coarse field -- concat vs interpolation, matched set.
# We never plot the no-TESSERA ConvCNP vs interp.
#
# BOTH ConvCNP variants evaluate on the SAME stations: the no-TESSERA runs carry
# --tessera-path purely as a station filter (tessera_injection=none), so they too
# are restricted to stations with a valid patch. Verified from the run summaries:
# europe 898 stations / 1,195,048 predictions for t2m and 550 / 687,073 for wind,
# identical for baseline and concat, and identical to the "matched" interpolation
# rows. The uplift-vs-model leg is therefore a like-for-like comparison.
# (The "full" interpolation rows -- 947 / 598 in europe -- predate that filter and
# now match NO model in this tree; the pre-filter baseline runs they were built to
# pair with survive only as *.fullset_bak directories. This cell uses "matched"
# throughout, which is the correct set for both variants.)
#
# The no-model reference is bilinear interpolation of the SAME 0.25 deg context
# field the model was given at that lead -- real ERA5 at lead 0, the Aurora
# forecast at +6/+24/+72h -- so it is not an "ERA5" reference beyond lead 0 and
# is not described as one. Bilinear interpolation is common to BOTH variables;
# the only thing that differs between them is the correction applied on top:
#   t2m   + a fitted lapse-rate correction. Plain interpolation leaves a
#         systematic elevation bias that a single fitted constant removes, so an
#         uplift measured against the uncorrected field credits the model for a
#         correction any trivial post-processing would make. Benchmarking against
#         the corrected floor is the honest (and harder) comparison.
#   wind  no correction. 10 m wind speed has no lapse analogue -- it does not
#         vary with height at a fixed environmental rate -- so no corrected
#         reference exists (and the baseline script refuses to build one).
# Falls back to the uncorrected field for t2m, with a warning, if the lapse rows
# are missing, so the cell still runs on a partial results tree.

CTX_REF_BY_TARGET = {"t2m": "interp_lapse", "wind": "interp"}
CTX_REF_EXTRA = {"interp": "", "interp_lapse": "+ fitted lapse rate"}
CTX_REF_KEY = "vs interpolated 0.25\u00b0 field"  # series key; legend label built below
CTX_REF_BASE = "vs bilinear interp of the 0.25\u00b0 context field"


def ctx_reference(target):
    """(rows, kind) — the interpolation reference this target is scored against, matched set."""
    want = CTX_REF_BY_TARGET.get(target, "interp")
    if want == "interp_lapse" and not INTERP_LAPSE.empty:
        f = INTERP_LAPSE[
            (INTERP_LAPSE.target == target) & (INTERP_LAPSE.station_set == "matched")
        ]
        if not f.empty:
            return f, "interp_lapse"
        print(
            f"[warn] no lapse-corrected rows for {target} — falling back to "
            "the uncorrected interpolated field."
        )
    if INTERP.empty:
        return INTERP, "interp"
    return INTERP[
        (INTERP.target == target) & (INTERP.station_set == "matched")
    ], "interp"


def uplift(frame, metric):
    """Per-seed TESSERA uplift over the no-TESSERA baseline (% better)."""
    keys = ["region", "target", "seed", "eval_label"]
    piv = frame.pivot_table(index=keys, columns="variant", values=metric, observed=True)
    if not {"baseline", "concat"}.issubset(piv.columns):
        return pd.DataFrame()
    sub = piv.dropna(subset=["baseline", "concat"])
    if sub.empty:
        return pd.DataFrame()
    out = sub.reset_index()[keys].copy()
    out["uplift_pct"] = (
        (sub["baseline"] - sub["concat"]) / sub["baseline"] * 100
    ).values
    return out


def tessera_uplift_summary(metric):
    """(region,target,eval_label,reference) -> mean/std uplift of TESSERA.

    The interpolation leg is built per target so each variable is scored against
    its own reference kind; `ref_kind` records which one was actually used.
    """
    recs = []
    u = uplift(D, metric)  # vs no-TESSERA (paired seeds)
    if not u.empty:
        a = (
            u.groupby(["region", "target", "eval_label"], observed=True)["uplift_pct"]
            .agg(mean="mean", std="std")
            .reset_index()
        )
        a["reference"] = "vs ConvCNP (no TESSERA)"
        a["ref_kind"] = ""
        recs.append(a)
    keys = ["region", "target", "eval_label"]
    for target in ["t2m", "wind"]:  # vs interp (matched set)
        ref, kind = ctx_reference(target)
        if ref.empty or metric not in ref.columns:
            continue
        mdl = (
            D[(D.variant == "concat") & (D.target == target)]
            .groupby(keys, observed=True)[metric]
            .agg(m="mean", s="std")
            .reset_index()
        )
        itp = ref[keys + [metric]].rename(columns={metric: "interp"})
        j = mdl.merge(itp, on=keys, how="inner")
        if j.empty:
            continue
        j["mean"] = (j["interp"] - j["m"]) / j["interp"] * 100
        j["std"] = j["s"] / j["interp"] * 100
        j["reference"] = CTX_REF_KEY
        j["ref_kind"] = kind
        recs.append(j[keys + ["mean", "std", "reference", "ref_kind"]])
    return pd.concat(recs, ignore_index=True) if recs else pd.DataFrame()


def ctx_legend_label(s):
    """'vs bilinear interp of the 0.25° context field (+ fitted lapse rate for t2m)'.

    Bilinear interpolation is common to both variables, so it is stated once and
    only the extra correction is qualified per variable. Built from what each
    target actually used rather than hard-coded, so the legend cannot claim a
    lapse correction the rows do not carry.
    """
    used = s[s.reference == CTX_REF_KEY].drop_duplicates(["target", "ref_kind"])[
        ["target", "ref_kind"]
    ]
    extras = [
        f"{CTX_REF_EXTRA[k]} for {t}"
        for t, k in zip(used["target"], used["ref_kind"], strict=False)
        if CTX_REF_EXTRA.get(k)
    ]
    return CTX_REF_BASE + (f" ({', '.join(extras)})" if extras else "")


# colour = reference; marker echoes the skill-vs-lead plots (interp = black diamond).
REF_STYLE = {
    "vs ConvCNP (no TESSERA)": dict(color="#1f77b4", marker="o"),
    CTX_REF_KEY: dict(color=INTERP_COLOR, marker="D"),
}


def plot_tessera_uplift(metric):
    s = tessera_uplift_summary(metric)
    if s.empty:
        print(f"(no {metric} uplift to plot)")
        return
    targets = [t for t in ["t2m", "wind"] if t in s["target"].unique()]
    nrow, ncol = len(targets), len(REGIONS)
    fig, axes = plt.subplots(
        nrow,
        ncol,
        figsize=(5.2 * ncol, 3.5 * nrow),
        squeeze=False,
        sharex=True,
        sharey="row",
    )
    x = np.arange(len(EVAL_ORDER))
    for ri, target in enumerate(targets):
        for ci, region in enumerate(REGIONS):
            ax = axes[ri][ci]
            for ref, st in REF_STYLE.items():
                cell = s[
                    (s.target == target) & (s.region == region) & (s.reference == ref)
                ]
                if cell.empty:
                    continue
                a = cell.set_index("eval_label").reindex(EVAL_ORDER)
                ax.errorbar(
                    x,
                    a["mean"].values,
                    yerr=a["std"].values,
                    marker=st["marker"],
                    color=st["color"],
                    lw=1.7,
                    ms=5,
                    capsize=3,
                    label=ref,
                )
            ax.axhline(0, color="k", lw=0.8)
            ax.axvline(
                0.5, color="k", ls=":", lw=0.8, alpha=0.5
            )  # lead-0 | forecast divider
            ax.set_xticks(x)
            ax.set_xticklabels(EVAL_ORDER, rotation=20, ha="right")
            if ri == 0:
                ax.set_title(region.replace("_", " ").title())
            if ci == 0:
                ax.set_ylabel(
                    f"{target}\n{metric.upper()} uplift (% better)", fontweight="bold"
                )
    handles = [
        Line2D(
            [0],
            [0],
            color=st["color"],
            marker=st["marker"],
            lw=1.8,
            ms=6,
            label=(ctx_legend_label(s) if ref == CTX_REF_KEY else ref),
        )
        for ref, st in REF_STYLE.items()
    ]
    # fig.suptitle(f"TESSERA uplift vs references — {metric.upper()}  "
    #              f"(reference model = ConvCNP with TESSERA)", y=0.99)
    fig.tight_layout(rect=[0, 0.08, 1, 0.96])
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(handles),
        fontsize=10,
        frameon=False,
        bbox_to_anchor=(0.5, 0.01),
    )
    fig.savefig(OUT_DIR / f"tessera_uplift_{metric}.png", bbox_inches="tight")
    plt.show()


for metric in ["rmse", "mae", "crps"]:
    plot_tessera_uplift(metric)

# --- Tables (TESSERA-referenced only) --------------------------------------
u = uplift(D, "rmse")
if not u.empty:
    tab = (
        u.groupby(["region", "target", "eval_label"], observed=True)["uplift_pct"]
        .agg(rmse_uplift_pct="mean", rmse_uplift_std="std", n_seeds="count")
        .reset_index()
    )
    print("RMSE uplift of TESSERA vs ConvCNP (no TESSERA) — % better, seed mean±std:")
    display(tab)
srmse = tessera_uplift_summary("rmse")
if not srmse.empty:
    vi = (
        srmse[srmse.reference == CTX_REF_KEY][
            ["region", "target", "ref_kind", "eval_label", "mean", "std"]
        ]
        .rename(columns={"mean": "uplift_%", "std": "uplift_std"})
        .sort_values(["target", "region", "eval_label"])
    )
    print(
        f"\nRMSE uplift of TESSERA vs the interpolated context field — % better "
        f"({ctx_legend_label(srmse)}):"
    )
    display(vi)

## 6. Calibration and the learned σ(lead)

This is where the lead-conditioned design differs most from zero-shot. The lead
channel lets the head **adapt its predictive spread to the lead**, so we expect:

- **mean predictive σ to rise with lead** (the head widens when the forecast
  context is less informative) — the direct readout of the learned σ(lead);
- **coverage to stay near nominal across leads** (68.3% within-1σ, 95.4%
  within-2σ) if that widening is well-calibrated.

Residual under-coverage that *grows* with lead would mean the lead channel only
partially taught the head to widen — a genuine model finding, not a frozen-spread
artifact (the zero-shot failure mode).

- **PIT χ²** statistic / p-value — uniformity of the probability-integral
  transform (higher χ², lower p = worse calibration).

In [ ]:
# === Calibration ===========================================================
def calib_table(frame, target):
    sub = frame[frame.target == target]
    if sub.empty:
        return pd.DataFrame()
    a = (
        sub.groupby(["region", "variant", "eval_label"], observed=True)
        .agg(
            within_1sigma=("within_1sigma", "mean"),
            within_2sigma=("within_2sigma", "mean"),
            pit_chi2=("pit_chi2_stat", "mean"),
            pit_p=("pit_chi2_pvalue", "mean"),
            mean_pred_std=("mean_pred_std", "mean"),
        )
        .reset_index()
    )
    return a


for target in ["t2m", "wind"]:
    t = calib_table(D, target)
    if t.empty:
        continue
    print(
        f"\n{target.upper()} calibration (nominal within-1\u03c3=68.3%, within-2\u03c3=95.4%):"
    )
    display(
        t.style.format(
            {
                "within_1sigma": "{:.1f}",
                "within_2sigma": "{:.1f}",
                "pit_chi2": "{:.1f}",
                "pit_p": "{:.3f}",
                "mean_pred_std": "{:.3f}",
            }
        ).hide(axis="index")
    )


# (a) mean predictive sigma vs lead -- the learned sigma(lead). Should rise.
def plot_pred_sigma_vs_lead(frame, target):
    sub = frame[frame.target == target]
    if sub.empty:
        return
    fig, axes = plt.subplots(
        1, len(REGIONS), figsize=(5.2 * len(REGIONS), 3.6), squeeze=False
    )
    x = np.arange(len(EVAL_ORDER))
    for ci, region in enumerate(REGIONS):
        ax = axes[0][ci]
        for variant in VARIANT_ORDER:
            cell = sub[(sub.region == region) & (sub.variant == variant)]
            a = (
                cell.groupby("eval_label", observed=True)["mean_pred_std"]
                .agg(["mean", "std"])
                .reindex(EVAL_ORDER)
            )
            if a["mean"].notna().sum() == 0:
                continue
            ax.plot(
                x,
                a["mean"].values,
                "-o",
                color=VARIANT_COLORS[variant],
                lw=1.8,
                ms=5,
                label=vlabel(variant),
            )
            lo = a["mean"].values - a["std"].fillna(0).values
            hi = a["mean"].values + a["std"].fillna(0).values
            ax.fill_between(x, lo, hi, color=VARIANT_COLORS[variant], alpha=0.15)
        ax.set_xticks(x)
        ax.set_xticklabels(EVAL_ORDER, rotation=20, ha="right")
        ax.set_title(f"{region} ({target})")
        ax.set_ylabel("mean predictive \u03c3")
        ax.legend(fontsize=8)
    fig.suptitle(
        f"{target.upper()} \u2014 learned \u03c3(lead): mean predictive spread vs lead",
        y=1.02,
    )
    fig.savefig(OUT_DIR / f"pred_sigma_vs_lead_{target}.png", bbox_inches="tight")
    plt.show()


for target in ["t2m", "wind"]:
    plot_pred_sigma_vs_lead(D, target)


# (b) coverage vs lead -- does the learned spread stay near nominal as lead grows?
def plot_coverage(frame, target):
    sub = frame[frame.target == target]
    if sub.empty:
        return
    fig, axes = plt.subplots(
        1, len(REGIONS), figsize=(5.4 * len(REGIONS), 3.8), squeeze=False
    )
    x = np.arange(len(EVAL_ORDER))
    for ci, region in enumerate(REGIONS):
        ax = axes[0][ci]
        for variant in VARIANT_ORDER:
            cell = sub[(sub.region == region) & (sub.variant == variant)]
            for col, ls in [("within_1sigma", "-"), ("within_2sigma", "--")]:
                a = (
                    cell.groupby("eval_label", observed=True)[col]
                    .mean()
                    .reindex(EVAL_ORDER)
                )
                if a.notna().sum() == 0:
                    continue
                ax.plot(
                    x,
                    a.values,
                    ls,
                    color=VARIANT_COLORS[variant],
                    lw=1.6,
                    label=f"{vlabel(variant)} {col.split('_')[1]}",
                )
        for nominal in (68.27, 95.45):
            ax.axhline(nominal, color="red", lw=0.7, alpha=0.6)
        ax.set_xticks(x)
        ax.set_xticklabels(EVAL_ORDER, rotation=20, ha="right")
        ax.set_title(f"{region} ({target})")
        ax.set_ylabel("empirical coverage %")
        ax.legend(fontsize=6, ncol=3)
    fig.suptitle(f"{target.upper()} coverage vs lead (red = nominal)", y=1.02)
    fig.savefig(OUT_DIR / f"coverage_{target}.png", bbox_inches="tight")
    plt.show()


for target in ["t2m", "wind"]:
    plot_coverage(D, target)

## 7. Seasonal breakdown

Seasonal MAE (point estimate = predictive mean) for a representative TESSERA
variant (`concat`; edit `SEASONAL_VARIANT` to switch). Does the longer-lead
penalty concentrate in particular seasons (e.g. convective JJA vs synoptic
DJF)?

In [ ]:
# === Seasonal MAE ==========================================================
SEASONAL_VARIANT = "concat"  # switch to "film" or "baseline" as needed
SEASONAL_VARIANT_DISP = VARIANT_DISPLAY.get(SEASONAL_VARIANT, SEASONAL_VARIANT)
seas_cols = [f"seasonal_mae_{s}" for s in SEASONS]


def plot_seasonal(frame, target):
    sub = frame[(frame.target == target) & (frame.variant == SEASONAL_VARIANT)]
    if sub.empty or sub[seas_cols].notna().sum().sum() == 0:
        print(f"(no seasonal data for {target} / {SEASONAL_VARIANT})")
        return
    fig, axes = plt.subplots(
        1, len(REGIONS), figsize=(5.6 * len(REGIONS), 3.6), squeeze=False
    )
    xs = np.arange(len(SEASONS))
    w = 0.2
    for ci, region in enumerate(REGIONS):
        ax = axes[0][ci]
        cell = sub[sub.region == region]
        for k, lab in enumerate(EVAL_ORDER):
            a = cell[cell.eval_label == lab][seas_cols].mean().values
            ax.bar(xs + (k - 1.5) * w, a, width=w, label=lab)
        ax.set_xticks(xs)
        ax.set_xticklabels(SEASONS)
        ax.set_title(f"{region} ({target}) \u2014 {SEASONAL_VARIANT_DISP}")
        ax.set_ylabel("seasonal MAE")
        ax.legend(fontsize=7)
    fig.suptitle(
        f"{target.upper()} seasonal MAE by lead ({SEASONAL_VARIANT_DISP})", y=1.02
    )
    fig.savefig(OUT_DIR / f"seasonal_{target}.png", bbox_inches="tight")
    plt.show()


for target in ["t2m", "wind"]:
    plot_seasonal(D, target)

## 8. Per-station spatial analysis (focus config)

Loads `test_station_errors.npz`. For a chosen config we map per-station MAE for
the lead-0 anchor vs the +72h lead and the **lead-72 − lead-0 map** (where the
longer-lead context hurts most), plus the per-station MAE distribution across
leads and whether the lead-72 increase correlates with **elevation / sub-grid
elevation** (complex terrain).

In [ ]:
# === Per-station ===========================================================
FOCUS = dict(region="europe", config=None, target="t2m", variant="concat", seed=42)


def focus_config_name():
    if FOCUS["config"]:
        return FOCUS["config"]
    cand = [
        c
        for c in df["config"].unique()
        if parse_config(c)["target"] == FOCUS["target"]
        and parse_config(c)["variant"] == FOCUS["variant"]
    ]
    return cand[0] if cand else None


def load_station_npz(region, config, seed, eval_key):
    f = (
        RESULTS_ROOT
        / region
        / f"{config}_seed{seed}"
        / eval_key
        / "test_station_errors.npz"
    )
    if not f.exists():
        return None
    return dict(np.load(f, allow_pickle=True))


cfg = focus_config_name()
var = FOCUS["target"]
print("Focus:", FOCUS["region"], cfg, f"seed{FOCUS['seed']}", "var", var)

st = {k: load_station_npz(FOCUS["region"], cfg, FOCUS["seed"], k) for k in EVAL_SOURCES}
st = {k: v for k, v in st.items() if v is not None}

if not st or cfg is None:
    print("No per-station npz found for the focus config \u2014 adjust FOCUS.")
else:
    ref = st.get("eval_lead0h") or next(iter(st.values()))
    lats, lons, elev = ref["station_lats"], ref["station_lons"], ref["station_elevs"]
    delev = ref.get("station_delta_elevs", np.full_like(elev, np.nan))
    cnt = ref[f"{var}_station_count"]
    valid = cnt > 0

    def mae_of(key):
        d = st.get(key)
        return d[f"{var}_station_mae"] if d is not None else None

    # (a) maps: lead 0, lead +72h, and delta
    a0 = mae_of("eval_lead0h")
    a72 = mae_of("eval_lead72h")
    panels = [("Lead 0 MAE", a0, "viridis"), ("Lead +72h MAE", a72, "viridis")]
    if a0 is not None and a72 is not None:
        panels.append(("\u0394 MAE (lead72 \u2212 lead0)", a72 - a0, "RdBu_r"))
    fig, axes = plt.subplots(
        1, len(panels), figsize=(5.2 * len(panels), 4.2), squeeze=False
    )
    for ax, (title, vals, cmap) in zip(axes[0], panels, strict=False):
        if vals is None:
            continue
        vlim = np.nanpercentile(np.abs(vals[valid]), 98) if "\u0394" in title else None
        sc = ax.scatter(
            lons[valid],
            lats[valid],
            c=vals[valid],
            s=14,
            cmap=cmap,
            vmin=(-vlim if vlim else None),
            vmax=(vlim if vlim else None),
        )
        plt.colorbar(sc, ax=ax, shrink=0.8)
        ax.set_title(title)
        ax.set_xlabel("lon")
        ax.set_ylabel("lat")
    fig.suptitle(f"Per-station {var} MAE \u2014 {FOCUS['region']} / {cfg}", y=1.03)
    fig.savefig(OUT_DIR / "station_maps.png", bbox_inches="tight")
    plt.show()

    # (b) per-station MAE distribution (ECDF) across leads
    fig, ax = plt.subplots(figsize=(6, 4))
    for key, (lab, _) in EVAL_SOURCES.items():
        v = mae_of(key)
        if v is None:
            continue
        vv = np.sort(v[valid])
        ax.plot(vv, np.linspace(0, 1, len(vv)), label=lab, lw=1.8)
    ax.set_xlabel(f"{var} per-station MAE")
    ax.set_ylabel("ECDF")
    ax.legend()
    ax.set_title("Per-station MAE distribution")
    fig.savefig(OUT_DIR / "station_mae_ecdf.png", bbox_inches="tight")
    plt.show()

    # (c) does the lead-72 increase track terrain?
    if a0 is not None and a72 is not None:
        dmae = (a72 - a0)[valid]
        fig, axx = plt.subplots(1, 2, figsize=(10, 3.8))
        for ax, xvar, name in [
            (axx[0], elev[valid], "elevation (m)"),
            (axx[1], delev[valid], "sub-grid \u0394elevation (m)"),
        ]:
            if np.all(np.isnan(xvar)):
                continue
            ax.scatter(xvar, dmae, s=10, alpha=0.5)
            good = np.isfinite(xvar) & np.isfinite(dmae)
            if good.sum() > 2:
                r = np.corrcoef(xvar[good], dmae[good])[0, 1]
                ax.set_title(f"\u0394MAE vs {name}  (r={r:.2f})")
            ax.set_xlabel(name)
            ax.set_ylabel("\u0394MAE (lead72\u2212lead0)")
            ax.axhline(0, color="k", lw=0.7)
        fig.savefig(OUT_DIR / "station_dmae_vs_terrain.png", bbox_inches="tight")
        plt.show()

## 9. Per-observation diagnostics (PIT, predicted-vs-observed, dispersion)

Loads `test_predictions.npz` (per-observation `mu`, `log_var`, targets). We
recompute the predictive CDF / mean self-contained — Gaussian for t2m,
Truncated-Normal on [0, ∞) for wind — matching `evaluate.py`'s ±10 log-var
clamp, and look across leads at: PIT histograms (flat = calibrated),
predicted-vs-observed, and the **dispersion ratio** `mean|err| / mean σ`.

For this lead-conditioned model the success signal is the dispersion ratio
staying ≈ 1 **even at +72h** — evidence the head used the lead channel to widen
σ in step with the growing error, rather than staying over-confident.

In [ ]:
# === Per-observation =======================================================
def load_pred_npz(region, config, seed, eval_key):
    f = (
        RESULTS_ROOT
        / region
        / f"{config}_seed{seed}"
        / eval_key
        / "test_predictions.npz"
    )
    if not f.exists():
        return None
    return dict(np.load(f, allow_pickle=True))


def predictive(distribution, mu, log_var):
    """Return (sigma, point_mean, cdf_fn) self-contained, matching evaluate.py clamp."""
    sigma = np.exp(0.5 * np.clip(log_var, -10.0, 10.0))
    if distribution == "gaussian":
        mean = mu
        cdf = lambda y: stats.norm.cdf(y, loc=mu, scale=sigma)
    elif distribution == "truncated_normal":  # lower truncation at 0 (wind >= 0)
        a = (0.0 - mu) / sigma
        mean = stats.truncnorm.mean(a, np.inf, loc=mu, scale=sigma)
        cdf = lambda y: stats.truncnorm.cdf(y, a, np.inf, loc=mu, scale=sigma)
    else:
        raise ValueError(distribution)
    return sigma, mean, cdf


dist = "gaussian" if var == "t2m" else "truncated_normal"
preds = {k: load_pred_npz(FOCUS["region"], cfg, FOCUS["seed"], k) for k in EVAL_SOURCES}
preds = {k: v for k, v in preds.items() if v is not None}

if not preds or cfg is None:
    print("No per-observation npz for the focus config \u2014 adjust FOCUS.")
else:
    # PIT histograms across leads.
    fig, axes = plt.subplots(
        1, len(preds), figsize=(3.4 * len(preds), 3.2), squeeze=False
    )
    for ax, (key, d) in zip(axes[0], preds.items(), strict=False):
        mu = d[f"{var}_param_mu"]
        lv = d[f"{var}_param_log_var"]
        y = d[f"{var}_targets"]
        _, _, cdf = predictive(dist, mu, lv)
        pit = cdf(y)
        pit = pit[np.isfinite(pit)]
        ax.hist(
            pit,
            bins=15,
            range=(0, 1),
            density=True,
            color="#1f77b4",
            alpha=0.8,
            edgecolor="white",
        )
        ax.axhline(1.0, color="red", lw=1)
        ax.set_title(EVAL_SOURCES[key][0])
        ax.set_xlabel("PIT")
        ax.set_ylim(0, None)
    fig.suptitle(f"PIT histograms \u2014 {var} ({cfg}). Flat = calibrated", y=1.04)
    fig.savefig(OUT_DIR / "pit_histograms.png", bbox_inches="tight")
    plt.show()

    # Predicted-vs-observed: lead 0 vs lead +72h.
    pairs = [("eval_lead0h", "ERA5 (lead 0)"), ("eval_lead72h", "Lead +72h")]
    pairs = [(k, l) for k, l in pairs if k in preds]
    if pairs:
        fig, axes = plt.subplots(
            1, len(pairs), figsize=(4.6 * len(pairs), 4.4), squeeze=False
        )
        for ax, (key, lab) in zip(axes[0], pairs, strict=False):
            d = preds[key]
            mu = d[f"{var}_param_mu"]
            lv = d[f"{var}_param_log_var"]
            y = d[f"{var}_targets"]
            _, mean, _ = predictive(dist, mu, lv)
            ax.hexbin(y, mean, gridsize=45, mincnt=1, cmap="viridis", bins="log")
            lo, hi = np.nanpercentile(np.concatenate([y, mean]), [1, 99])
            ax.plot([lo, hi], [lo, hi], "r--", lw=1)
            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)
            ax.set_xlabel(f"observed {var}")
            ax.set_ylabel(f"predicted {var}")
            ax.set_title(lab)
        fig.suptitle(f"Predicted vs observed \u2014 {var} ({cfg})", y=1.02)
        fig.savefig(OUT_DIR / "pred_vs_obs.png", bbox_inches="tight")
        plt.show()

    # Dispersion: predictive sigma distribution + mean|err|/sigma per lead.
    fig, axx = plt.subplots(1, 2, figsize=(11, 3.8))
    for key, d in preds.items():
        sigma, mean, _ = predictive(
            dist, d[f"{var}_param_mu"], d[f"{var}_param_log_var"]
        )
        axx[0].hist(
            sigma,
            bins=40,
            histtype="step",
            density=True,
            lw=1.6,
            label=EVAL_SOURCES[key][0],
        )
    axx[0].set_xlabel("predictive \u03c3")
    axx[0].set_ylabel("density")
    axx[0].legend(fontsize=8)
    axx[0].set_title("Sharpness (predictive \u03c3) by lead")
    rows_disp = []
    for key, d in preds.items():
        sigma, mean, _ = predictive(
            dist, d[f"{var}_param_mu"], d[f"{var}_param_log_var"]
        )
        err = np.abs(mean - d[f"{var}_targets"])
        rows_disp.append(
            (
                EVAL_SOURCES[key][0],
                np.mean(err),
                np.mean(sigma),
                np.mean(err) / np.mean(sigma),
            )
        )
    dd = pd.DataFrame(
        rows_disp, columns=["lead", "mean|err|", "mean \u03c3", "ratio |err|/\u03c3"]
    )
    dd = dd.set_index("lead").reindex(EVAL_ORDER).dropna(how="all")
    axx[1].bar(
        np.arange(len(dd)), dd["ratio |err|/\u03c3"].values, color="#1f77b4", alpha=0.85
    )
    axx[1].axhline(1.0, color="k", lw=1, ls="--")
    axx[1].set_xticks(np.arange(len(dd)))
    axx[1].set_xticklabels(dd.index, rotation=20, ha="right")
    axx[1].set_ylabel("mean |err| / mean \u03c3")
    axx[1].set_title("Dispersion (\u22481 = well-dispersed; >1 = under-dispersed)")
    fig.savefig(OUT_DIR / "dispersion.png", bbox_inches="tight")
    plt.show()
    print("Dispersion summary (focus config):")
    display(dd)

## 9b. Spread–skill ratio, rank histogram & PIT reliability (pooled over all seeds + regions)

The reviewer's two questions — *"did you compute the spread–skill ratio?"* and
*"a rank histogram to check the obs fall ~uniformly across the predicted
percentiles?"* — are answered here, pooled across **every** seed and region for
statistical power.

- **Spread–skill ratio** `SSR = RMS(σ) / RMSE`, per lead. For a *parametric*
  predictive distribution the ideal is **exactly 1** (no finite-ensemble
  correction needed). `SSR < 1` ⇒ over-confident (spread too small), `SSR > 1` ⇒
  over-dispersed. The lead-conditioned question is whether SSR stays flat at ≈1
  **as lead grows** — i.e. whether the learned σ(lead) widens *fast enough* to
  keep pace with the rising RMSE rather than the spread freezing.
- **Rank (PIT) histogram** — the continuous-distribution analogue of the
  ensemble rank histogram: bin the probability-integral transform `F(y)` and
  check for uniformity. Flat = calibrated; ∪-shaped = under-dispersed;
  ∩-shaped = over-dispersed; a tilt = systematic bias.
- **PIT reliability** — the empirical PIT CDF against the diagonal; the same
  information as a calibration/reliability curve.

Point estimate = predictive **mean** (the functional the spread–skill ratio is
defined against), so wind uses the truncated-normal mean, not μ. Truncated-normal
moments and CDF use closed forms — analytic == scipy to 1e-15 in-regime but
~1800× faster, which is what makes the full-set pass feasible.

In [ ]:
# === Pooled spread-skill & PIT: streaming pass over every prediction file ===
# Aggregates sufficient statistics only (no raw arrays kept), so it scales to
# the full ~1.3M-obs-per-cell set across all seeds + regions. Per
# (region, target, variant, seed, lead) we keep n, Σσ² and Σerr² (→ RMS spread,
# RMSE, spread-skill ratio) plus a PIT histogram. Truncated-normal moments/CDF
# use closed forms (analytic == scipy to 1e-15 in-regime, ~1800× faster), with
# evaluate.py's ±10 log-var clamp. Point estimate = predictive *mean*.
from collections import defaultdict

from scipy.stats import norm

NB_PIT = 20
LEAD_BY_LABEL = {lab: lead for (lab, lead) in EVAL_SOURCES.values()}


def _gauss_stats(mu, sigma, y):
    return mu, sigma, norm.cdf((y - mu) / sigma)


def _tn_stats(mu, sigma, y):
    """Lower-truncated normal at 0 → (predictive mean, std, PIT), all analytic."""
    # Log-domain forms (norm.logcdf) — STABLE for μ/σ ≪ 0 (calm wind), where the
    # naive 1−Φ(−μ/σ) underflows and collapses the predictive mean onto μ (<0).
    s = mu / sigma  # = -alpha
    logZ = norm.logcdf(s)  # log P(X > 0)
    lam = np.exp((-0.5 * s * s - 0.5 * np.log(2.0 * np.pi)) - logZ)  # φ(α)/Z, stable
    mean = mu + sigma * lam
    var = sigma**2 * np.clip(1.0 - s * lam - lam**2, 0.0, None)  # σ²(1+αλ−λ²)
    pit = np.where(
        y >= 0.0,
        np.clip(-np.expm1(norm.logcdf((mu - y) / sigma) - logZ), 0.0, 1.0),
        0.0,
    )
    return mean, np.sqrt(var), pit


def _stats_for(target, mu, log_var, y):
    """(predictive mean, predictive std, PIT) for a head, matching evaluate.py."""
    sigma = np.exp(0.5 * np.clip(log_var, -10.0, 10.0))
    return _gauss_stats(mu, sigma, y) if target == "t2m" else _tn_stats(mu, sigma, y)


acc = defaultdict(lambda: dict(n=0, svar=0.0, sse=0.0, hist=np.zeros(NB_PIT)))
nfiles = 0
for region in REGIONS:
    region_dir = RESULTS_ROOT / region
    if not region_dir.is_dir():
        continue
    for seed_dir in sorted(region_dir.glob("*_seed*")):
        m = re.match(r"(?P<config>.+)_seed(?P<seed>\d+)$", seed_dir.name)
        if not m:
            continue
        config, seed = m.group("config"), int(m.group("seed"))
        fac = parse_config(config)
        if (
            fac["variant"] not in VARIANT_ORDER
        ):  # skip variants we don't compare (e.g. film)
            continue
        target = fac["target"]
        for eval_key, (lead_label, lead) in EVAL_SOURCES.items():
            f = seed_dir / eval_key / "test_predictions.npz"
            if not f.exists():
                continue
            with np.load(f, allow_pickle=True) as d:
                mu = d[f"{target}_param_mu"].astype(np.float64)
                lv = d[f"{target}_param_log_var"].astype(np.float64)
                y = d[f"{target}_targets"].astype(np.float64)
            mean, std, pit = _stats_for(target, mu, lv, y)
            err = mean - y
            ok = np.isfinite(err) & np.isfinite(std)
            a = acc[(region, target, fac["variant"], seed, lead_label)]
            a["n"] += int(ok.sum())
            a["svar"] += float(np.sum(std[ok] ** 2))
            a["sse"] += float(np.sum(err[ok] ** 2))
            a["hist"] += np.histogram(
                pit[np.isfinite(pit)], bins=NB_PIT, range=(0.0, 1.0)
            )[0]
            nfiles += 1
    print(f"  ...{region}: cumulative {nfiles} files")
print(f"Done. Pooled {nfiles} prediction files.")

# Tidy per-seed spread-skill table (one row per region/target/variant/seed/lead).
recs = []
for (region, target, variant, seed, lead_label), a in acc.items():
    if a["n"] == 0:
        continue
    rms_spread = np.sqrt(a["svar"] / a["n"])
    rmse = np.sqrt(a["sse"] / a["n"])
    recs.append(
        dict(
            region=region,
            target=target,
            variant=variant,
            seed=seed,
            eval_label=lead_label,
            lead_hours=LEAD_BY_LABEL[lead_label],
            rms_spread=rms_spread,
            rmse=rmse,
            ssr=rms_spread / rmse,
            n=a["n"],
        )
    )
ssr_df = pd.DataFrame(recs)
ssr_df["eval_label"] = pd.Categorical(
    ssr_df["eval_label"], categories=EVAL_ORDER, ordered=True
)

# PIT histogram pooled across seeds + regions, per (target, variant, lead).
pit_pool = defaultdict(lambda: np.zeros(NB_PIT))
for (region, target, variant, seed, lead_label), a in acc.items():
    pit_pool[(target, variant, lead_label)] += a["hist"]
print(f"ssr_df rows: {len(ssr_df)}")

In [ ]:
# === Spread-skill ratio, rank histogram & PIT reliability (plots) ==========
# (a) SSR table, (b) SSR-vs-lead curve, (c) pooled rank/PIT histogram,
# (d) PIT reliability. Together: "does the learned σ grow fast enough to track
# the true error as lead increases?" — SSR flat at ≈1 + flat PIT at every lead
# == yes.

# (a) SSR table (seed mean±std).
for target in ["t2m", "wind"]:
    sub = ssr_df[ssr_df.target == target]
    if sub.empty:
        continue
    tab = (
        sub.groupby(["region", "variant", "eval_label"], observed=True)["ssr"]
        .agg(["mean", "std"])
        .reset_index()
    )
    tab["cell"] = [
        fmt_ms(mn, sd) for mn, sd in zip(tab["mean"], tab["std"], strict=False)
    ]
    piv = tab.pivot_table(
        index=["region", "variant"],
        columns="eval_label",
        values="cell",
        aggfunc="first",
        observed=False,
    ).reindex(columns=EVAL_ORDER)
    print(
        f"\n{target.upper()} spread-skill ratio  RMS(σ)/RMSE  "
        f"(1.00 = calibrated; <1 over-confident, >1 over-dispersed):"
    )
    display(piv)


# (b) SSR vs lead.
def plot_ssr_vs_lead(target):
    sub = ssr_df[ssr_df.target == target]
    if sub.empty:
        return
    fig, axes = plt.subplots(
        1, len(REGIONS), figsize=(5.2 * len(REGIONS), 3.7), squeeze=False
    )
    x = np.arange(len(EVAL_ORDER))
    for ci, region in enumerate(REGIONS):
        ax = axes[0][ci]
        for variant in VARIANT_ORDER:
            cell = sub[(sub.region == region) & (sub.variant == variant)]
            a = (
                cell.groupby("eval_label", observed=True)["ssr"]
                .agg(["mean", "std"])
                .reindex(EVAL_ORDER)
            )
            if a["mean"].notna().sum() == 0:
                continue
            ax.errorbar(
                x,
                a["mean"].values,
                yerr=a["std"].fillna(0).values,
                fmt="-o",
                color=VARIANT_COLORS[variant],
                lw=1.8,
                ms=5,
                capsize=3,
                label=vlabel(variant),
            )
        ax.axhline(1.0, color="red", lw=0.9, ls="--")
        ax.set_xticks(x)
        ax.set_xticklabels(EVAL_ORDER, rotation=20, ha="right")
        ax.set_ylabel("spread-skill ratio  RMS(σ)/RMSE")
        ax.set_title(f"{region} ({target})")
        ax.legend(fontsize=8)
    fig.suptitle(
        f"{target.upper()} — spread-skill ratio vs lead  "
        f"(=1 calibrated; flat ⇒ σ tracks error growth)",
        y=1.02,
    )
    fig.savefig(OUT_DIR / f"spread_skill_ratio_{target}.png", bbox_inches="tight")
    plt.show()


for target in ["t2m", "wind"]:
    plot_ssr_vs_lead(target)

# (c) Pooled rank (PIT) histogram — rows = variant, cols = lead.
centers = (np.arange(NB_PIT) + 0.5) / NB_PIT
width = 1.0 / NB_PIT
for target in ["t2m", "wind"]:
    variants = [
        v
        for v in VARIANT_ORDER
        if any(
            pit_pool.get((target, v, l), np.zeros(NB_PIT)).sum() > 0 for l in EVAL_ORDER
        )
    ]
    if not variants:
        continue
    fig, axes = plt.subplots(
        len(variants),
        len(EVAL_ORDER),
        figsize=(2.7 * len(EVAL_ORDER), 2.4 * len(variants)),
        squeeze=False,
    )
    for ri, variant in enumerate(variants):
        for cj, lead_label in enumerate(EVAL_ORDER):
            ax = axes[ri][cj]
            h = pit_pool.get((target, variant, lead_label), np.zeros(NB_PIT))
            if h.sum() == 0:
                ax.axis("off")
                continue
            dens = h / h.sum() / width
            ax.bar(
                centers,
                dens,
                width=width * 0.95,
                color=VARIANT_COLORS[variant],
                alpha=0.85,
                edgecolor="white",
            )
            ax.axhline(1.0, color="red", lw=1)
            ax.set_ylim(0, max(1.6, dens.max() * 1.18))
            if ri == 0:
                ax.set_title(lead_label, fontsize=9)
            if cj == 0:
                ax.set_ylabel(f"{vlabel(variant)}\nPIT density", fontsize=8)
            ax.set_xlabel("PIT", fontsize=8)
            ax.tick_params(labelsize=7)
    fig.suptitle(
        f"{target.upper()} — rank (PIT) histogram, pooled seeds+regions  "
        f"(flat=calibrated, ∪=under-dispersed, ∩=over-dispersed, slope=bias)",
        y=1.005,
    )
    fig.savefig(OUT_DIR / f"rank_histogram_{target}.png", bbox_inches="tight")
    plt.show()

# (d) PIT reliability (empirical PIT CDF vs uniform; on the diagonal = calibrated).
edges = np.linspace(0.0, 1.0, NB_PIT + 1)
for target in ["t2m", "wind"]:
    if not any(
        pit_pool.get((target, v, l), np.zeros(NB_PIT)).sum() > 0
        for v in VARIANT_ORDER
        for l in EVAL_ORDER
    ):
        continue
    fig, axes = plt.subplots(
        1, len(VARIANT_ORDER), figsize=(4.6 * len(VARIANT_ORDER), 4.1), squeeze=False
    )
    colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(EVAL_ORDER)))
    for ci, variant in enumerate(VARIANT_ORDER):
        ax = axes[0][ci]
        for li, lead_label in enumerate(EVAL_ORDER):
            h = pit_pool.get((target, variant, lead_label), np.zeros(NB_PIT))
            if h.sum() == 0:
                continue
            emp = np.concatenate([[0.0], np.cumsum(h) / h.sum()])
            ax.plot(edges, emp, "-o", ms=3, lw=1.5, color=colors[li], label=lead_label)
        ax.plot([0, 1], [0, 1], "k--", lw=1)
        ax.set_xlabel("nominal quantile")
        ax.set_ylabel("empirical PIT CDF")
        ax.set_title(f"{target} — {vlabel(variant)}")
        ax.legend(fontsize=7)
    fig.suptitle(
        f"{target.upper()} — PIT reliability (on diagonal = calibrated)", y=1.02
    )
    fig.savefig(OUT_DIR / f"pit_reliability_{target}.png", bbox_inches="tight")
    plt.show()

### 9c. Spread–skill *relationship* diagram (focus config)

The scalar SSR above checks calibration *on average*. This checks **resolution**:
within each lead we bin observations by their predicted spread and plot per-bin
RMSE against per-bin RMS spread. Points tracking the 1:1 line mean the model's σ
is heteroscedastic in a useful way — a larger predicted σ really does coincide
with a larger realised error (the model knows *when* it is uncertain), not just
right on average. A flat cloud well below the diagonal would mean σ carries
little case-by-case information.

In [ ]:
# === Spread-skill relationship diagram (focus config, pooled seeds) ========
# Bin observations by predicted spread (deciles); compare per-bin RMS spread (x)
# to per-bin RMSE (y). On the 1:1 line ⇒ predicted σ is not just right on
# average but *resolves* which forecasts are genuinely harder. Reuses the fast
# analytic _stats_for and load_pred_npz / FOCUS / cfg / var from §9 and §9b.
dist_target = FOCUS["target"]
NQ = 10
if cfg is None:
    print("No focus config resolved — set FOCUS in §8.")
else:
    fig, axes = plt.subplots(
        1, len(EVAL_SOURCES), figsize=(3.5 * len(EVAL_SOURCES), 3.4), squeeze=False
    )
    for ax, (eval_key, (lead_label, lead)) in zip(
        axes[0], EVAL_SOURCES.items(), strict=False
    ):
        mus, lvs, ys = [], [], []
        for seed in SEEDS:
            d = load_pred_npz(FOCUS["region"], cfg, seed, eval_key)
            if d is None:
                continue
            mus.append(d[f"{dist_target}_param_mu"])
            lvs.append(d[f"{dist_target}_param_log_var"])
            ys.append(d[f"{dist_target}_targets"])
        if not mus:
            ax.axis("off")
            ax.set_title(f"{lead_label}\n(no data)")
            continue
        mu = np.concatenate(mus)
        lv = np.concatenate(lvs)
        y = np.concatenate(ys).astype(np.float64)
        mean, std, _ = _stats_for(dist_target, mu, lv, y)
        err = mean - y
        ok = np.isfinite(std) & np.isfinite(err)
        std, err = std[ok], err[ok]
        qs = np.quantile(std, np.linspace(0.0, 1.0, NQ + 1))
        qs[-1] += 1e-9
        idx = np.clip(np.digitize(std, qs[1:-1]), 0, NQ - 1)
        bx, by = [], []
        for b in range(NQ):
            sel = idx == b
            if sel.sum() < 50:
                continue
            bx.append(np.sqrt(np.mean(std[sel] ** 2)))  # RMS spread in bin
            by.append(np.sqrt(np.mean(err[sel] ** 2)))  # RMSE in bin
        bx, by = np.array(bx), np.array(by)
        ax.plot(bx, by, "-o", color="#1f77b4", lw=1.6, ms=5)
        hi = (
            float(
                np.nanmax([bx.max() if bx.size else 1.0, by.max() if by.size else 1.0])
            )
            * 1.05
        )
        ax.plot([0, hi], [0, hi], "r--", lw=1)
        ax.set_xlim(0, hi)
        ax.set_ylim(0, hi)
        ax.set_xlabel("binned RMS spread")
        ax.set_ylabel("binned RMSE")
        ax.set_title(lead_label)
    fig.suptitle(
        f"Spread-skill relationship — {dist_target} ({cfg}); on 1:1 = calibrated & resolving",
        y=1.04,
    )
    fig.savefig(OUT_DIR / "spread_skill_relationship.png", bbox_inches="tight")
    plt.show()

### 9d. Publication calibration synthesis — sample-size-robust error, PIT-based coverage, paper figures

Turns §9/§9b into the two paper figures + the paper table for the *lead-conditioned*
model, across **lead × region × variant × variable**, pooled over all seeds for
statistical power. Both heads are fully parametric, so every calibration statistic
is a closed form of the predictive CDF (Gaussian $\Phi(\tfrac{y-\mu}{\sigma})$;
left-truncated-Normal $(\Phi(\tfrac{y-\mu}{\sigma})-\Phi(\alpha))/(1-\Phi(\alpha))$,
$\alpha=-\mu/\sigma$) — no ensemble/sampling — which is what makes the exact
full-set pass feasible.

Three ideas make this paper-grade rather than the raw diagnostics above:

- **Sample-size-robust calibration error.** At $N\!\sim\!10^6$ the PIT $\chi^2$
  $p$-value is always $\approx 0$ and the $\chi^2$ *magnitude* scales with $N$
  (so Europe's $\sim\!10^4$–$10^5$ isn't comparable to East Asia's $\sim\!10^2$–$10^4$
  — it just has more stations). We instead report the **total-variation distance of
  the PIT histogram from uniform**, $\mathrm{CE_{TV}}=\tfrac12\sum_b|\hat p_b-1/B|\in[0,1]$
  (and the RMS reliability deviation). Bounded, $N$-independent, comparable across
  every cell — a calibration *effect size*.
- **PIT-based central-interval coverage.** Coverage of the nominal central-$q$
  interval $=$ fraction with $|\mathrm{PIT}-\tfrac12|<q/2$. Derived through the PIT it
  is exact and identical in meaning for **both** heads, unlike the "$|y-\mu|<k\sigma$"
  proxy `evaluate.py` reports, which uses the *untruncated* $\sigma$ and overstates
  wind coverage (~76–81% vs the corrected ~69–72%).
- **True predictive std in the SSR.** For the truncated Normal the spread is the
  closed-form $\sigma_\mathrm{pred}^2=\sigma^2[1+\alpha\lambda-\lambda^2]$,
  $\lambda=\varphi(\alpha)/(1-\Phi(\alpha))$ — not the raw $\sigma$.

**Outputs** (→ `OUT_DIR`, copy into the paper's `figures/`): `calib_vs_lead.png`
(coverage / SSR / $\mathrm{CE_{TV}}$ vs lead, rows = variable), `pit_rank_hist.png`
(PIT rank histograms of the concat model, variable × lead), `calibration_summary.csv`,
and LaTeX rows for `tab:calib`.


In [ ]:
# === 9d. Publication calibration synthesis (compute + paper figures) ========
# One streaming pass over every prediction file, reusing the fast analytic
# `_stats_for` from §9b (Gaussian / truncated-Normal closed forms, evaluate.py's
# ±10 log-var clamp). Per (region, target, variant, seed, lead) we keep the
# sufficient statistics for: spread-skill ratio, PIT-based central-interval
# coverage (exact for both heads), and the sample-size-robust PIT calibration
# error CE_TV. CRPS is read straight from test_summary.json (its wind value uses
# the ensemble estimator, which the npz doesn't store).
from collections import defaultdict

NB_PIT = 20  # PIT histogram resolution
# Half-widths c s.t. |PIT-0.5|<c is the nominal central interval:
#   68.27% -> 0.34134, 95.45% -> 0.47725.
C1, C2 = 0.3413447461, 0.4772498681
LEAD_BY_LABEL = {lab: lead for (lab, lead) in EVAL_SOURCES.values()}

if "_stats_for" not in dir():  # allow running §9d without §9b
    from scipy.stats import norm

    def _stats_for(target, mu, log_var, y):
        sigma = np.exp(0.5 * np.clip(log_var, -10.0, 10.0))
        if target == "t2m":
            return mu, sigma, norm.cdf((y - mu) / sigma)
        s = mu / sigma  # stable (μ/σ ≪ 0 calm wind)
        logZ = norm.logcdf(s)
        lam = np.exp((-0.5 * s * s - 0.5 * np.log(2.0 * np.pi)) - logZ)
        mean = mu + sigma * lam
        var = sigma**2 * np.clip(1.0 - s * lam - lam**2, 0.0, None)
        pit = np.where(
            y >= 0.0,
            np.clip(-np.expm1(norm.logcdf((mu - y) / sigma) - logZ), 0.0, 1.0),
            0.0,
        )
        return mean, np.sqrt(var), pit


cacc = defaultdict(
    lambda: dict(n=0, svar=0.0, sse=0.0, hist=np.zeros(NB_PIT), c1=0, c2=0)
)
crps_by = {}  # (region,target,variant,seed,lead_label) -> crps
pitpool = defaultdict(
    lambda: np.zeros(NB_PIT)
)  # (target,variant,lead_label) pooled seeds+regions
for region in REGIONS:
    region_dir = RESULTS_ROOT / region
    if not region_dir.is_dir():
        continue
    for seed_dir in sorted(region_dir.glob("*_seed*")):
        m = re.match(r"(?P<config>.+)_seed(?P<seed>\d+)$", seed_dir.name)
        if not m:
            continue
        fac = parse_config(m.group("config"))
        if fac["variant"] not in VARIANT_ORDER:
            continue
        target, seed = fac["target"], int(m.group("seed"))
        for eval_key, (lead_label, lead) in EVAL_SOURCES.items():
            f = seed_dir / eval_key / "test_predictions.npz"
            if not f.exists():
                continue
            with np.load(f, allow_pickle=True) as d:
                mu = d[f"{target}_param_mu"].astype(np.float64)
                lv = d[f"{target}_param_log_var"].astype(np.float64)
                y = d[f"{target}_targets"].astype(np.float64)
            mean, std, pit = _stats_for(target, mu, lv, y)
            err = mean - y
            ok = np.isfinite(err) & np.isfinite(std) & np.isfinite(pit)
            key = (region, target, fac["variant"], seed, lead_label)
            a = cacc[key]
            a["n"] += int(ok.sum())
            a["svar"] += float(np.sum(std[ok] ** 2))
            a["sse"] += float(np.sum(err[ok] ** 2))
            h = np.histogram(pit[ok], bins=NB_PIT, range=(0.0, 1.0))[0]
            a["hist"] += h
            a["c1"] += int(np.sum(np.abs(pit[ok] - 0.5) < C1))
            a["c2"] += int(np.sum(np.abs(pit[ok] - 0.5) < C2))
            pitpool[(target, fac["variant"], lead_label)] += h
            summ = seed_dir / eval_key / "test_summary.json"
            if summ.exists():
                crps_by[key] = g(json.loads(summ.read_text()), f"{target}_crps")


def _ce_scores(hist):
    """(CE_TV, CE_RMSCE) from a PIT histogram — both -> 0 when calibrated."""
    p = hist / hist.sum()
    ce_tv = 0.5 * np.abs(p - 1.0 / len(hist)).sum()  # total variation, [0,1]
    emp = np.cumsum(p)
    nom = np.arange(1, len(hist) + 1) / len(hist)
    ce_rmsce = float(np.sqrt(np.mean((emp - nom) ** 2)))  # RMS reliability dev
    return ce_tv, ce_rmsce


recs = []
for (region, target, variant, seed, lead_label), a in cacc.items():
    if a["n"] == 0:
        continue
    ce_tv, ce_rmsce = _ce_scores(a["hist"])
    recs.append(
        dict(
            region=region,
            target=target,
            variant=variant,
            seed=seed,
            eval_label=lead_label,
            lead_hours=LEAD_BY_LABEL[lead_label],
            n=a["n"],
            ssr=np.sqrt(a["svar"] / a["n"]) / np.sqrt(a["sse"] / a["n"]),
            cov68=100 * a["c1"] / a["n"],
            cov95=100 * a["c2"] / a["n"],
            ce_tv=ce_tv,
            ce_rmsce=ce_rmsce,
            crps=crps_by.get((region, target, variant, seed, lead_label), np.nan),
        )
    )
calib_df = pd.DataFrame(recs)
calib_df["eval_label"] = pd.Categorical(
    calib_df["eval_label"], categories=EVAL_ORDER, ordered=True
)
print(f"calib_df rows: {len(calib_df)}  (per region×target×variant×seed×lead)")

# ---- Figure 1: calibration vs lead (rows = variable, cols = metric) --------
# Paper-facing names: concat is the sole TESSERA variant, so label the two
# variants as the ConvCNP with / without TESSERA rather than by fusion mechanism.
VARIANT_DISPLAY = {"baseline": "ConvCNP (no TESSERA)", "concat": "ConvCNP with TESSERA"}
REGION_DISPLAY = {"europe": "Europe", "east_asia": "East Asia"}
REGION_LS = {"europe": "-", "east_asia": "--"}
REGION_MK = {"europe": "o", "east_asia": "s"}
METRICS = [
    ("cov68", "central 68.3% coverage (%)", 68.27, (58, 78)),
    ("ssr", "spread-skill ratio  RMS(σ)/RMSE", 1.0, (0.80, 1.10)),
    ("ce_tv", "PIT calibration error  CE$_{TV}$", 0.0, (0.0, 0.13)),
]
xh = np.array([LEAD_BY_LABEL[l] for l in EVAL_ORDER])
fig, axes = plt.subplots(2, 3, figsize=(12, 6.4), sharex=True, squeeze=False)
for ri, target in enumerate(["t2m", "wind"]):
    for ci, (col, title, nominal, ylim) in enumerate(METRICS):
        ax = axes[ri][ci]
        for region in REGIONS:
            for variant in VARIANT_ORDER:
                cell = calib_df[
                    (calib_df.region == region)
                    & (calib_df.target == target)
                    & (calib_df.variant == variant)
                ]
                a = (
                    cell.groupby("eval_label", observed=True)[col]
                    .agg(["mean", "std"])
                    .reindex(EVAL_ORDER)
                )
                if a["mean"].notna().sum() == 0:
                    continue
                ax.errorbar(
                    xh,
                    a["mean"].values,
                    yerr=a["std"].fillna(0).values,
                    color=VARIANT_COLORS[variant],
                    ls=REGION_LS[region],
                    marker=REGION_MK[region],
                    ms=5,
                    lw=1.7,
                    capsize=2,
                )
        ax.axhline(nominal, color="k", lw=0.9, ls=":")
        ax.set_ylim(*ylim)
        ax.set_xticks(xh)
        if ri == 0:
            ax.set_title(title, fontsize=10)
        if ci == 0:
            ax.set_ylabel(target, fontsize=12, fontweight="bold")
        if ri == 1:
            ax.set_xlabel("forecast lead (h)")
# One standalone legend beneath all panels (colour = model, line style = region),
# so the panels stay uncluttered.
_vh = [
    Line2D([0], [0], color=VARIANT_COLORS[v], lw=2.8, label=VARIANT_DISPLAY[v])
    for v in VARIANT_ORDER
]
_rh = [
    Line2D(
        [0],
        [0],
        color="0.35",
        ls=REGION_LS[r],
        marker=REGION_MK[r],
        lw=1.8,
        ms=6,
        label=REGION_DISPLAY[r],
    )
    for r in REGIONS
]
fig.suptitle(
    "Lead-conditioned model — calibration vs forecast lead "
    "(pooled seeds+stations; dotted = ideal)",
    y=0.99,
)
fig.tight_layout(rect=[0, 0.075, 1, 0.97])
fig.legend(
    handles=_vh + _rh,
    loc="lower center",
    ncol=4,
    fontsize=9.5,
    frameon=False,
    bbox_to_anchor=(0.5, 0.005),
    columnspacing=2.2,
    handlelength=2.4,
)
fig.savefig(OUT_DIR / "calib_vs_lead.png", bbox_inches="tight")
plt.show()

# ---- Figure 2: PIT rank histograms, concat model, variable × lead ----------
centers = (np.arange(NB_PIT) + 0.5) / NB_PIT
width = 1.0 / NB_PIT
fig, axes = plt.subplots(
    2,
    len(EVAL_ORDER),
    figsize=(2.75 * len(EVAL_ORDER), 5.2),
    sharey=True,
    squeeze=False,
)
for ri, target in enumerate(["t2m", "wind"]):
    for ci, lead_label in enumerate(EVAL_ORDER):
        ax = axes[ri][ci]
        h = pitpool.get((target, "concat", lead_label), np.zeros(NB_PIT))
        if h.sum() == 0:
            ax.axis("off")
            continue
        ax.bar(
            centers,
            h / h.sum() / width,
            width=width * 0.95,
            color=VARIANT_COLORS["concat"],
            alpha=0.85,
            edgecolor="white",
            lw=0.3,
        )
        ax.axhline(1.0, color="k", lw=1, ls=":")
        ax.set_ylim(0, 1.7)
        if ri == 0:
            ax.set_title(lead_label, fontsize=10)
        if ci == 0:
            ax.set_ylabel(f"{target}\nPIT density", fontsize=10, fontweight="bold")
        if ri == 1:
            ax.set_xlabel("PIT")
fig.suptitle(
    "Lead-conditioned ConvCNP with TESSERA — PIT rank histograms "
    "(flat = calibrated; pooled seeds+regions)",
    y=1.0,
)
fig.tight_layout()
fig.savefig(OUT_DIR / "pit_rank_hist.png", bbox_inches="tight")
plt.show()

In [ ]:
# === 9d (cont.) Publication calibration table + CSV + LaTeX rows ============
# Seed-mean per (variable, region, variant, lead) for the headline metrics, plus
# the compact lead-0 vs +72h view that goes into tab:calib.
def _calib_pivot(target, col, fmt="{:.3f}"):
    sub = calib_df[calib_df.target == target]
    a = (
        sub.groupby(["region", "variant", "eval_label"], observed=True)[col]
        .mean()
        .reset_index()
    )
    p = a.pivot_table(
        index=["region", "variant"],
        columns="eval_label",
        values=col,
        aggfunc="first",
        observed=False,
    ).reindex(columns=EVAL_ORDER)
    return p.style.format(fmt)


for target in ["t2m", "wind"]:
    if (calib_df.target == target).sum() == 0:
        continue
    print(
        f"\n{'=' * 66}\n{target.upper()} — central-68.3% coverage % "
        f"(nominal 68.27; PIT-based, exact for both heads)\n{'=' * 66}"
    )
    display(_calib_pivot(target, "cov68", "{:.1f}"))
    print(f"{target.upper()} — spread-skill ratio RMS(σ)/RMSE (ideal 1.000)")
    display(_calib_pivot(target, "ssr", "{:.3f}"))
    print(f"{target.upper()} — PIT calibration error CE_TV (ideal 0; N-robust)")
    display(_calib_pivot(target, "ce_tv", "{:.4f}"))
    print(f"{target.upper()} — CRPS (proper score; wind = ensemble estimator)")
    display(_calib_pivot(target, "crps", "{:.3f}"))

# Tidy CSV for the record (seed mean ± std, every metric/cell).
summary = (
    calib_df.groupby(["target", "region", "variant", "eval_label"], observed=True)
    .agg(
        cov68=("cov68", "mean"),
        cov68_std=("cov68", "std"),
        cov95=("cov95", "mean"),
        ssr=("ssr", "mean"),
        ssr_std=("ssr", "std"),
        ce_tv=("ce_tv", "mean"),
        ce_rmsce=("ce_rmsce", "mean"),
        crps=("crps", "mean"),
    )
    .reset_index()
)
summary.to_csv(OUT_DIR / "calibration_summary.csv", index=False)
print("\nWrote", OUT_DIR / "calibration_summary.csv")

# LaTeX rows for tab:calib (lead-0 anchor vs +72h extreme).
REG_TEX = {"europe": "Europe", "east_asia": "E.\\,Asia"}
VAR_TEX = {"baseline": "ConvCNP (no \\tessera)", "concat": "ConvCNP w/ \\tessera"}
L0, L72 = EVAL_ORDER[0], EVAL_ORDER[-1]


def _m(target, region, variant, lead, col):
    v = summary[
        (summary.target == target)
        & (summary.region == region)
        & (summary.variant == variant)
        & (summary.eval_label == lead)
    ][col]
    return v.iloc[0] if len(v) else float("nan")


print("\n% ---- tab:calib rows (paste into the table body) ----")
for target in ["t2m", "wind"]:
    print(f"\\multicolumn{{8}}{{l}}{{\\emph{{{target}}}}}\\\\")
    for region in REGIONS:
        for variant in VARIANT_ORDER:
            vals = [
                _m(target, region, variant, L0, "cov68"),
                _m(target, region, variant, L72, "cov68"),
                _m(target, region, variant, L0, "ssr"),
                _m(target, region, variant, L72, "ssr"),
                _m(target, region, variant, L0, "ce_tv"),
                _m(target, region, variant, L72, "ce_tv"),
            ]
            print(
                f"{REG_TEX[region]} & {VAR_TEX[variant]} & "
                f"{vals[0]:.1f} & {vals[1]:.1f} & {vals[2]:.3f} & {vals[3]:.3f} & "
                f"{vals[4]:.3f} & {vals[5]:.3f}\\\\"
            )

## 10. Publication summary table + CSV export

A compact table (rows = target / variant; columns = region × lead) of the
primary metric as `mean ± std`, plus a dump of the full tidy table and the
headline pivots to CSV under `cross_lead_analysis_outputs/`.

In [ ]:
# === Summary table + export ================================================
def publication_table(frame, metric):
    a = agg(frame, metric)
    a["cell"] = [fmt_ms(m, s) for m, s in zip(a["mean"], a["std"], strict=False)]
    a["col"] = a["region"].astype(str) + " / " + a["eval_label"].astype(str)
    col_order = [f"{r} / {e}" for r in REGIONS for e in EVAL_ORDER]
    p = a.pivot_table(
        index=["target", "variant"],
        columns="col",
        values="cell",
        aggfunc="first",
        observed=False,
    )
    return p.reindex(columns=[c for c in col_order if c in p.columns])


for metric in ["rmse", "mae", "crps"]:
    print(
        f"\n{'=' * 80}\nPUBLICATION TABLE \u2014 {metric.upper()} (mean\u00b1std over seeds)\n{'=' * 80}"
    )
    pt = publication_table(D, metric)
    display(pt)
    pt.to_csv(OUT_DIR / f"summary_{metric}.csv")

# Full tidy table for the record.
df.to_csv(OUT_DIR / "all_results_tidy.csv", index=False)
print("\nWrote:", sorted(p.name for p in OUT_DIR.glob("*.csv")))
print("Figures:", sorted(p.name for p in OUT_DIR.glob("*.png")))

### Caveats

- **n = 3 seeds.** Error bands are ±1 std over three seeds — indicative, not a
  significance test. Treat small variant gaps with care.
- **Lead-conditioned, not zero-shot.** Each model was trained jointly on all four
  leads with a `lead/72` channel, and each eval sets `--lead-hours` to match.
  So "skill vs lead" (§3) is the genuine forecast-context information-loss curve,
  and the predictive spread is *expected* to widen with lead (the learned
  σ(lead), §6/§9). Calibration holding across leads is the success criterion;
  under-dispersion that grows with lead means the lead channel only partially
  taught the head to widen — a model finding, not the zero-shot frozen-spread
  artifact.
- **Lead 0 = real ERA5 analysis** (the shortest-lead anchor), in-distribution for
  the lead-0 slice of training; the +6/+24/+72h leads are Aurora forecasts.
- **Point estimate.** For the Truncated-Normal (wind) head the point estimate is
  the truncated mean, recomputed self-contained in §9 to match `evaluate.py`.
- Edit `RESULTS_ROOT` (cell 2), `FOCUS` (§8) and `SEASONAL_VARIANT` (§7) to
  retarget; re-run top-to-bottom.